# Prompt → Policy: train a MuJoCo policy from a natural-language reward

Companion notebook for the *Finding Theta* blog post. The pipeline:

1. **An LLM** turns a natural-language description into a JSON reward spec.
2. **PPO** trains a policy in one of three MuJoCo envs against that reward.
3. We render a deterministic rollout video of the trained policy.

**Pick an env:** `halfcheetah` (2D, two legs, fast forward locomotion), `hopper` (2D, *one foot*, hopping/balance), `ant` (3D, four legs, walking/turning).

**Pick a provider:**
- `anthropic` — Claude API. Needs `ANTHROPIC_API_KEY`.
- `gemini` — Google Gemini API. Needs `GEMINI_API_KEY`.
- `local` — quantized open-source model on the Colab GPU. No API key. Needs enough VRAM (T4 works in 4-bit; L4/A100 are smoother).

**Recommended runtime:** *Runtime → Change runtime type → A100* (or L4) for the local-LLM path. T4 is fine for the API paths and works for local in 4-bit but is tight on VRAM.

## 1. Install

All extras installed by default so you can switch providers without reinstalling. Set `INSTALL_*` flags to `False` to skip an extra you don't need.

In [ ]:
BRANCH = "main"  # replace with a branch/tag/commit pin if needed
INSTALL_LOCAL = True   # transformers/accelerate/bitsandbytes for --provider local
INSTALL_GEMINI = True  # google-genai for --provider gemini

_extras = []
if INSTALL_LOCAL:
    _extras.append("local")
if INSTALL_GEMINI:
    _extras.append("gemini")
_extras_str = f"[{','.join(_extras)}]" if _extras else ""

!pip install --quiet "prompt-to-policy{_extras_str} @ git+https://github.com/kuds/rl-llm-reward.git@{BRANCH}"

## 2. API keys and runtime setup

Add `ANTHROPIC_API_KEY` and/or `GEMINI_API_KEY` as Colab secrets (Tools → Secrets → New secret) if you plan to use those providers. Skip this if you're only using `local`.

In [ ]:
import os

# MuJoCo needs an OpenGL backend for rgb_array rendering. Colab supports EGL out of the box.
os.environ.setdefault("MUJOCO_GL", "egl")

_in_colab = False
try:
    from google.colab import userdata
    _in_colab = True
    for _key in ("ANTHROPIC_API_KEY", "GEMINI_API_KEY", "GOOGLE_API_KEY"):
        try:
            v = userdata.get(_key)
        except Exception:
            v = None
        if v:
            os.environ[_key] = v
except ImportError:
    pass

_have_anthropic = bool(os.environ.get("ANTHROPIC_API_KEY"))
_have_gemini = bool(os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY"))
print(f"ANTHROPIC_API_KEY  : {'set' if _have_anthropic else 'NOT set'}")
print(f"GEMINI_API_KEY     : {'set' if _have_gemini else 'NOT set'}")

# Quick GPU diagnostic — useful when deciding whether `local` is viable.
try:
    import torch
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        free, total = torch.cuda.mem_get_info()
        print(f"GPU                : {name} ({free/1e9:.1f}/{total/1e9:.1f} GB free)")
    else:
        print("GPU                : none (CPU only). Use --provider anthropic/gemini.")
except Exception as e:
    print(f"GPU                : couldn't query torch ({e})")

## 3. Quick smoke test (no API key, hand-written reward)

Confirms the env, harness, and video rendering work. ~2–3 minutes on a T4. 10k steps is far below the recommended budget — expect a wobbly gait, not a fast run.

In [ ]:
from pathlib import Path

from prompt_to_policy.reward import RewardSpec
from prompt_to_policy.train import train

quick_spec = RewardSpec.model_validate({
    "components": [
        {"feature": "forward_velocity", "weight": 1.0},
        {"feature": "control_cost", "weight": -0.05},
    ],
})

quick_result = train(
    spec=quick_spec,
    prompt="(quick test, hand-written) run forward",
    env="halfcheetah",
    timesteps=10_000,
    run_dir=Path("runs/colab_quick"),
    seed=0,
    record_video=True,
)

print(f"final mean return : {quick_result.final_mean_return:.1f}")
print(f"mean ep length    : {quick_result.final_mean_length:.0f}")
print(f"train seconds     : {quick_result.train_seconds:.1f}")
print(f"video             : {quick_result.video_path}")

In [ ]:
from IPython.display import Video

Video(str(quick_result.video_path), embed=True, width=480)

## 4. Configure your run

Edit `ENV`, `PROVIDER`, and `PROMPT` to taste, then run the next two cells.

**Some prompts that work well:**
- HalfCheetah: `"run forward as fast as possible"`, `"run backward"`, `"stand still and stay upright"`
- Hopper: `"hop forward as fast as possible without falling"`, `"hop in place as high as you can"`, `"stay perfectly still and balanced on one foot"`
- Ant: `"walk forward steadily"`, `"spin clockwise in place"`, `"move sideways without rotating"`

In [ ]:
ENV = "hopper"           # "halfcheetah" | "hopper" | "ant"
PROVIDER = "anthropic"   # "anthropic" | "gemini" | "local"
PROMPT = "hop forward as fast as possible without falling"
TIMESTEPS = 200_000      # bump to 1_000_000 for full-quality policies

# Provider-specific knobs.
ANTHROPIC_MODEL = "claude-opus-4-7"
GEMINI_MODEL = "gemini-2.5-pro"
LOCAL_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
LOCAL_QUANTIZATION = "4bit"  # "4bit" (T4-friendly) | "8bit" | "none" (A100 / big GPU)

In [ ]:
from prompt_to_policy import envs
from prompt_to_policy.llm import (
    GeminiRewardClient,
    LLMRewardClient,
    LocalLLMRewardClient,
)

env_spec = envs.get(ENV)
_no_key_msg = "set the corresponding API key as a Colab secret or switch PROVIDER"

if PROVIDER == "anthropic":
    if not os.environ.get("ANTHROPIC_API_KEY"):
        raise RuntimeError(f"ANTHROPIC_API_KEY not set — {_no_key_msg}.")
    client = LLMRewardClient(
        feature_docs=env_spec.feature_docs,
        model=ANTHROPIC_MODEL,
        cache_dir=Path("colab_llm_cache"),
        build_prompt=env_spec.build_system_prompt,
        env_name=ENV,
    )
elif PROVIDER == "gemini":
    if not (os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")):
        raise RuntimeError(f"GEMINI_API_KEY not set — {_no_key_msg}.")
    client = GeminiRewardClient(
        feature_docs=env_spec.feature_docs,
        model=GEMINI_MODEL,
        cache_dir=Path("colab_llm_cache"),
        build_prompt=env_spec.build_system_prompt,
        env_name=ENV,
    )
elif PROVIDER == "local":
    client = LocalLLMRewardClient(
        feature_docs=env_spec.feature_docs,
        model_id=LOCAL_MODEL_ID,
        quantization=LOCAL_QUANTIZATION,
        cache_dir=Path("colab_local_cache"),
        build_prompt=env_spec.build_system_prompt,
        env_name=ENV,
    )
else:
    raise ValueError(f"unknown PROVIDER {PROVIDER!r}")

gen = client.generate(PROMPT)

print(f"env       : {ENV}")
print(f"provider  : {PROVIDER} ({gen.model})")
print(f"prompt    : {PROMPT!r}")
print(f"cached    : {gen.cached}")
print(f"cost_usd  : {gen.estimated_cost_usd:.4f}")
print(f"spec      : {gen.spec.model_dump_json()}")

## 5. Train PPO and render the rollout

Free the LLM weights before training if you ran `local` on a smaller GPU — PPO + MuJoCo + a 7B model in 4-bit is tight on a 16 GB T4. The cell below does this automatically when `PROVIDER == 'local'`.

In [ ]:
import gc

import torch

if PROVIDER == "local":
    del client
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

result = train(
    spec=gen.spec,
    prompt=PROMPT,
    env=ENV,
    timesteps=TIMESTEPS,
    run_dir=Path("runs/colab_e2e"),
    seed=0,
    record_video=True,
)

print(f"final mean return : {result.final_mean_return:.1f}")
print(f"mean ep length    : {result.final_mean_length:.0f}")
print(f"train seconds     : {result.train_seconds:.1f}")
print(f"video             : {result.video_path}")

In [ ]:
Video(str(result.video_path), embed=True, width=480)

## Where to go from here

- Edit `PROMPT`, `ENV`, or `PROVIDER` and re-run sections 4 and 5. The on-disk cache means re-running the same `(env, provider, prompt)` triple is free.
- Compare what happens when the LLM produces a reward whose intent matches the prompt vs. one that doesn't — that's the *good* failure mode the post is about. Claude, Gemini, and a 7B local model often disagree in interesting ways.
- Bump `TIMESTEPS` to `1_000_000` for full-quality policies. Plan for ~30 min on a T4, ~10 min on an A100.
- The full feature registries, prompt templates, and PPO hyperparameters live in `src/prompt_to_policy/`. Hyperparameters are fixed per env; only the reward changes per prompt.